In [4]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from stockfish import Stockfish

# ==========================================================
# 設定
# ==========================================================
STOCKFISH_PATH = "D:/lichess2/UndergraduationProject/stockfish/stockfish-windows-x86-64-avx2.exe"
REP_GAMES_DIR = "D:/lichess2/UndergraduationProject/representative_games"

# ==========================================================
# 建立 Stockfish (depth=5)
# ==========================================================
def create_stockfish():
    sf = Stockfish(path=STOCKFISH_PATH, depth=5)
    sf.update_engine_parameters({
        "Threads": 2,
        "Minimum Thinking Time": 30
    })
    return sf

# ==========================================================
# 清理 Moves 欄位（移除 move number）
# ==========================================================
def convert_moves_for_stockfish(moves_str):
    if pd.isna(moves_str):
        return []
    cleaned_moves = re.sub(r'\d+\.\.\.|(\d+\.)', '', moves_str)
    moves = cleaned_moves.strip().split()
    return [m for m in moves if m]

# ==========================================================
# 計算每一步的評估值
# ==========================================================
def get_all_evaluations(stockfish: Stockfish, moves_list):
    evals = []
    for i in range(1, len(moves_list) + 1):
        stockfish.set_position(moves_list[:i])
        eval_info = stockfish.get_evaluation()
        if eval_info["type"] == "cp":
            val = eval_info["value"]
        elif eval_info["type"] == "mate":
            val = 10000 if eval_info["value"] > 0 else -10000
        else:
            val = 0
        evals.append(val)
    return evals

# ==========================================================
# 主流程
# ==========================================================
def evaluate_representative_games():
    sf = create_stockfish()
    total_files = 0
    start = time.time()

    for elo_range in sorted(os.listdir(REP_GAMES_DIR)):
        elo_folder = os.path.join(REP_GAMES_DIR, elo_range)
        if not os.path.isdir(elo_folder):
            continue

        print(f"\n📁 處理分數區間：{elo_range}")
        for player_id in os.listdir(elo_folder):
            player_folder = os.path.join(elo_folder, player_id)
            if not os.path.isdir(player_folder):
                continue

            for fname in os.listdir(player_folder):
                if not fname.endswith("_games.csv"):
                    continue

                file_path = os.path.join(player_folder, fname)
                total_files += 1

                df = pd.read_csv(file_path)
                moves_col = "MovesProcessed" if "MovesProcessed" in df.columns else "Moves"

                if moves_col not in df.columns:
                    print(f"⚠️ 檔案缺少 {moves_col} 欄位：{file_path}")
                    continue

                if "Evaluation_Depth_5" not in df.columns:
                    df["Evaluation_Depth_5"] = None

                for i, row in tqdm(
                    df.iterrows(), total=len(df),
                    desc=f"🔹 {elo_range}/{player_id}", leave=False
                ):
                    if pd.notna(row.get("Evaluation_Depth_5")):
                        continue
                    moves_list = convert_moves_for_stockfish(row[moves_col])
                    if not moves_list:
                        continue

                    evals = get_all_evaluations(sf, moves_list)
                    eval_str = ", ".join(f"{j+1}. {val}" for j, val in enumerate(evals))
                    df.at[i, "Evaluation_Depth_5"] = eval_str

                df.to_csv(file_path, index=False, encoding="utf-8-sig")
                print(f"✅ 完成 {file_path}")

    print(f"\n🎯 全部分數區間處理完成，共 {total_files} 個玩家檔案")
    print(f"⏱️ 總耗時：{time.time() - start:.2f} 秒")

# ==========================================================
# 執行
# ==========================================================
if __name__ == "__main__":
    evaluate_representative_games()



📁 處理分數區間：1000-1099


✅ 完成 D:/lichess2/UndergraduationProject/representative_games\1000-1099\abdullah_azad\abdullah_azad_games.csv


✅ 完成 D:/lichess2/UndergraduationProject/representative_games\1000-1099\Abdulm0hsen\Abdulm0hsen_games.csv


✅ 完成 D:/lichess2/UndergraduationProject/representative_games\1000-1099\adriangurrola\adriangurrola_games.csv


✅ 完成 D:/lichess2/UndergraduationProject/representative_games\1000-1099\Amirhoseyn_87\Amirhoseyn_87_games.csv


...

✅ 完成 D:/lichess2/UndergraduationProject/representative_games\900-999\twixy122\twixy122_games.csv


✅ 完成 D:/lichess2/UndergraduationProject/representative_games\900-999\VasilisDoulas\VasilisDoulas_games.csv


✅ 完成 D:/lichess2/UndergraduationProject/representative_games\900-999\What22022\What22022_games.csv


✅ 完成 D:/lichess2/UndergraduationProject/representative_games\900-999\willbefine1\willbefine1_games.csv


✅ 完成 D:/lichess2/UndergraduationProject/representative_games\900-999\Woukou\Woukou_games.csv

🎯 全部分數區間處理完成，共 1037 個玩家檔案
⏱️ 總耗時：5227.78 秒


In [5]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import numpy as np
import os

# ======================================================
# 🧠 設定：手動指定棋局 CSV 路徑
# ======================================================
GAME_PATH = r"代表棋局的路徑.csv"        # ← 你自己填
SCORES_PATH = r"同局的分數結果檔.csv"     # ← 你自己填
PLAYER_ID = "user1234"                     # ← 你要分析的代表玩家ID
THRESHOLD = 3                              # clock變化門檻（秒）

# ======================================================
# 🔍 函式：解析 Clock
# ======================================================
def parse_clock(clock_str):
    """解析 Clock 欄位，回傳 (white_times, black_times)"""
    if pd.isna(clock_str) or not isinstance(clock_str, str):
        return [], []

    pattern = re.findall(r'(\d+)\. (\d+:\d+:\d+)|(\d+)\.\.\. (\d+:\d+:\d+)', clock_str)
    white_times, black_times = [], []

    for p in pattern:
        if p[0]:  # 白方
            h, m, s = map(int, p[1].split(":"))
            white_times.append(h*3600 + m*60 + s)
        elif p[2]:  # 黑方
            h, m, s = map(int, p[3].split(":"))
            black_times.append(h*3600 + m*60 + s)
    return white_times, black_times

# ======================================================
# 📈 函式：分析時間變化與評分變化
# ======================================================
def analyze_time_score_relation(times, scores, threshold=3):
    """分析 Δtime > threshold 時對應的分數變化"""
    if len(times) <= 1 or len(scores) != len(times):
        return pd.DataFrame(columns=["Move", "TimeChange", "ScoreDiff"])

    time_diffs = np.diff(times) * -1  # 正值表示花的時間
    avg_score = np.mean(scores)
    score_diffs = np.array(scores) - avg_score

    # 找出時間變化超過門檻的步
    significant = np.where(time_diffs > threshold)[0] + 1
    data = []
    for i in significant:
        data.append({
            "Move": i,
            "TimeChange": time_diffs[i-1],
            "ScoreDiff": score_diffs[i]
        })
    return pd.DataFrame(data), time_diffs, score_diffs, avg_score

# ======================================================
# 🧩 主流程
# ======================================================
# 讀取棋局與分數
game_df = pd.read_csv(GAME_PATH)
score_df = pd.read_csv(SCORES_PATH)

clock_str = game_df["Clock"].iloc[0]
white_id = game_df["White"].iloc[0]
black_id = game_df["Black"].iloc[0]
scores = score_df["Score"].tolist()

white_times, black_times = parse_clock(clock_str)
is_white = (white_id == PLAYER_ID)
side = "White" if is_white else "Black"

side_times = white_times if is_white else black_times

# 分析
result_df, time_diffs, score_diffs, avg_score = analyze_time_score_relation(side_times, scores, THRESHOLD)

# ======================================================
# 📊 視覺化
# ======================================================
moves = np.arange(1, len(side_times) + 1)

fig, ax1 = plt.subplots(figsize=(12, 6))
ax2 = ax1.twinx()

# 時間線
ax1.plot(moves, side_times, color="blue", label=f"{side} Remaining Time (s)")
ax1.set_xlabel("Move Number")
ax1.set_ylabel("Remaining Time (s)", color="blue")
ax1.tick_params(axis='y', labelcolor="blue")

# 評分線
ax2.plot(moves, scores, color="red", label="Stockfish Evaluation", alpha=0.7)
ax2.axhline(avg_score, color="gray", linestyle="--", label=f"Avg Eval ({avg_score:.1f})")
ax2.set_ylabel("Evaluation (cp)", color="red")
ax2.tick_params(axis='y', labelcolor="red")

# 標出時間變化明顯的點
for _, row in result_df.iterrows():
    move = row["Move"]
    ax1.axvline(move, color="orange", linestyle="--", alpha=0.4)
    ax2.text(move, scores[move-1], f"Δt={row['TimeChange']:.1f}s\nΔE={row['ScoreDiff']:.1f}",
             rotation=45, fontsize=8, color="darkorange")

plt.title(f"⏳ Time & Evaluation Change for {PLAYER_ID} ({side})\nThreshold: {THRESHOLD}s")
fig.tight_layout()
plt.legend(loc="upper right")
plt.grid(True)
plt.show()

# ======================================================
# 🧾 印出結果
# ======================================================
print("\n📋 明顯時間變化步 (Δt > 3s):")
print(result_df)


In [4]:
analyze_player_game(
    "D:/lichess2/UndergraduationProject/representative_games/1000-1099/abdullah_azad/abdullah_azad_games.csv",
    n=3
)

ValueError: array length 286 does not match index length 2